# Boston Community Intervention — Crew & Catalyst Network Dashboard

Interactive map matching the KC StoryMap visual style, adapted for Boston data.

### Data files (place in same directory as notebook)
- `Fake_Small_BostonCrews_Networks.xlsx` — crew/catalyst data
- `2020_Census_Tracts_in_Boston.csv` — census tract IDs

### Requirements
```
pip install pandas folium requests branca shapely openpyxl geopandas
```


In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import json
import random
import math
import requests
import branca.colormap as cm
from shapely.geometry import shape, Point
from datetime import datetime
from pathlib import Path

## 1. Scrape Shooting Data from Boston PD

This cell downloads the **latest shooting CSV** directly from
[Analyze Boston](https://data.boston.gov/dataset/shootings).
The dataset updates weekly with a 7-day rolling delay.

Set `USE_LOCAL = True` to skip the download and use a previously saved CSV.

In [3]:
USE_LOCAL = False
LOCAL_CSV = Path("Data Sourcing/Boston/data/boston_shootings.csv")
# Stable Analyze Boston CKAN resource ID for Shootings
RESOURCE_ID = "73c7e069-701f-4910-986d-b950f46c91a1"
# Stable CKAN API endpoint
DATASTORE_SEARCH_URL = "https://data.boston.gov/api/3/action/datastore_search"

def download_boston_shootings_from_api():
    """
    Downloads all Boston shootings records directly from the Analyze Boston CKAN API.
    This avoids relying on temporary CSV download links like tmp...csv.
    """
    all_records = []
    limit = 5000
    offset = 0

    while True:
        params = {
            "resource_id": RESOURCE_ID,
            "limit": limit,
            "offset": offset
        }

        response = requests.get(DATASTORE_SEARCH_URL, params=params, timeout=60)
        response.raise_for_status()

        result = response.json()["result"]
        records = result["records"]

        if len(records) == 0:
            break

        all_records.extend(records)

        offset += limit

        if offset >= result["total"]:
            break

    df = pd.DataFrame(all_records)

    return df


if USE_LOCAL:
    print("⟳ Loading Boston shooting data from local CSV …")
    df_shoot = pd.read_csv(LOCAL_CSV)

else:
    print("⟳ Downloading latest Boston shooting data from Analyze Boston API …")

    try:
        df_shoot = download_boston_shootings_from_api()

        # Save updated copy locally so future fallback still works
        LOCAL_CSV.parent.mkdir(parents=True, exist_ok=True)
        df_shoot.to_csv(LOCAL_CSV, index=False)

        print(f"  ✓ Downloaded {len(df_shoot):,} rows from API")
        print(f"  ✓ Saved updated copy to {LOCAL_CSV}")

    except Exception as e:
        print(f"  ✗ API download failed: {e}")
        print("    Falling back to local CSV …")

        df_shoot = pd.read_csv(LOCAL_CSV)

# Clean / parse date column
df_shoot["shooting_date"] = pd.to_datetime(df_shoot["shooting_date"], utc=True, errors="coerce")

# Optional: drop rows where shooting_date failed to parse
df_shoot = df_shoot.dropna(subset=["shooting_date"])

print(f"\nLoaded {len(df_shoot):,} incidents")
print(
    f"Date range: {df_shoot['shooting_date'].min().strftime('%Y-%m-%d')} → "
    f"{df_shoot['shooting_date'].max().strftime('%Y-%m-%d')}"
)
print(
    f"Fatal: {(df_shoot['shooting_type_v2'] == 'Fatal').sum():,}  |  "
    f"Non-Fatal: {(df_shoot['shooting_type_v2'] == 'Non-Fatal').sum():,}")

⟳ Downloading latest Boston shooting data from Analyze Boston API …
  ✓ Downloaded 2,183 rows from API
  ✓ Saved updated copy to Data Sourcing\Boston\data\boston_shootings.csv

Loaded 2,183 incidents
Date range: 2015-01-01 → 2026-04-13
Fatal: 361  |  Non-Fatal: 1,822


## 2. Load Crew & Catalyst Network

In [4]:
CREW_PATH = "Fake_Small_BostonCrews_Networks.xlsx"

df_crews = pd.read_excel(CREW_PATH)
df_crews["Crew"] = df_crews["Crew"].str.strip().str.title()

def split_lines(val):
    if pd.isna(val):
        return []
    return [x.strip() for x in str(val).replace(",", "\n").split("\n") if x.strip()]

catalyst_to_crew = {}
crew_catalysts = {}
for _, row in df_crews.iterrows():
    crew = row["Crew"]
    cats = split_lines(row["Catalysts"])
    crew_catalysts[crew] = cats
    for c in cats:
        catalyst_to_crew[c] = (crew, row["Lat"], row["Long"])

crew_names_lower = {c.lower(): c for c in df_crews["Crew"]}
def resolve_crew(name):
    return crew_names_lower.get(name.strip().lower()) if name else None

crew_affiliations, crew_conflicts = [], []
for _, row in df_crews.iterrows():
    crew = row["Crew"]
    for other in split_lines(row["Affiliations/Collaborations"]):
        r = resolve_crew(other)
        if r and r != crew:
            pair = tuple(sorted([crew, r]))
            if pair not in crew_affiliations:
                crew_affiliations.append(pair)
    for other in split_lines(row["Conflicts w Other Crews"]):
        r = resolve_crew(other)
        if r and r != crew:
            pair = tuple(sorted([crew, r]))
            if pair not in crew_conflicts:
                crew_conflicts.append(pair)

catalyst_edges = []
for _, row in df_crews.iterrows():
    crew = row["Crew"]
    cats_here = crew_catalysts[crew]
    for tgt in split_lines(row["Catalst Network"]):
        if tgt in catalyst_to_crew and cats_here:
            pair = tuple(sorted([cats_here[0], tgt]))
            if pair not in catalyst_edges and pair[0] != pair[1]:
                catalyst_edges.append(pair)

print(f"Crews: {len(df_crews)}  |  Catalysts: {len(catalyst_to_crew)}")
print(f"Affiliations: {len(crew_affiliations)}  |  "
      f"Conflicts: {len(crew_conflicts)}  |  "
      f"Catalyst links: {len(catalyst_edges)}")

FileNotFoundError: [Errno 2] No such file or directory: 'Fake_Small_BostonCrews_Networks.xlsx'

## 3. Load 2020 Census Tract Boundaries

The local CSV has tract IDs/centroids. Polygon geometry comes from
the U.S. Census Bureau's cartographic boundary shapefile at:

`https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_25_tract_500k.zip`

This is a stable, permanent URL hosted by the Census Bureau.

In [ ]:
df_tracts_csv = pd.read_csv("2020_Census_Tracts_in_Boston.csv")
boston_geoids = set(df_tracts_csv["geoid20"].astype(str))
print(f"Boston tract IDs from CSV: {len(boston_geoids)}")

TIGER_URL = (
    "https://www2.census.gov/geo/tiger/GENZ2020/shp/"
    "cb_2020_25_tract_500k.zip"
)

print("Downloading Census Bureau tract polygons for Massachusetts …")
gdf_all = gpd.read_file(TIGER_URL)
print(f"  Total MA tracts: {len(gdf_all)}")

# Filter to Suffolk County (FIPS 025 = Boston)
gdf_tracts = gdf_all[gdf_all["COUNTYFP"] == "025"].copy()
if len(gdf_tracts) == 0:
    # Fallback: match by GEOID against our CSV
    gdf_tracts = gdf_all[gdf_all["GEOID"].isin(boston_geoids)].copy()

gdf_tracts["tract_id"] = gdf_tracts["GEOID"].astype(str)
gdf_tracts = gdf_tracts.to_crs("EPSG:4326")
print(f"  Boston tracts matched: {len(gdf_tracts)}")

## 4. Assign Shootings to Census Tracts

In [ ]:
DISTRICT_COORDS = {
    "A1":  (42.3580, -71.0580), "A7":  (42.3690, -71.0390),
    "A15": (42.3270, -71.0470), "B2":  (42.3290, -71.0840),
    "B3":  (42.3010, -71.0800), "C6":  (42.3380, -71.0380),
    "C11": (42.2990, -71.0610), "D4":  (42.3440, -71.0770),
    "D14": (42.3530, -71.1320), "E5":  (42.3050, -71.1150),
    "E13": (42.3230, -71.1040), "E18": (42.2830, -71.1520),
}

random.seed(42)
shoot_points = []
for _, row in df_shoot.iterrows():
    dist = row["district"]
    if dist not in DISTRICT_COORDS:
        continue
    lat, lon = DISTRICT_COORDS[dist]
    jlat = lat + random.gauss(0, 0.008)
    jlon = lon + random.gauss(0, 0.009)
    shoot_points.append({
        "geometry": Point(jlon, jlat),
        "incident": 1,
        "fatal": row["shooting_type_v2"] == "Fatal",
    })

gdf_points = gpd.GeoDataFrame(shoot_points, crs="EPSG:4326")

joined = gpd.sjoin(gdf_points, gdf_tracts, how="left", predicate="within")
tract_counts = joined.groupby("tract_id")["incident"].sum().fillna(0)

gdf_tracts["shoot_count"] = (
    gdf_tracts["tract_id"].map(tract_counts).fillna(0).astype(int)
)

print(f"Incidents mapped to tracts: {int(gdf_tracts['shoot_count'].sum()):,}")
print(f"Tracts with ≥1 incident: {(gdf_tracts['shoot_count'] > 0).sum()}")
print(f"Max per tract: {gdf_tracts['shoot_count'].max()}")

## 5. Build the Map

Visual style matched to the KC StoryMap:
- **Green-to-red choropleth** (green = 0, yellow = low, orange = mid, red = high)
- **Orange graduated circles** for crews (sized by catalyst count)
- **Light basemap** with thin grey tract borders
- **Left-side legend panel**

In [ ]:
# ── Base map ──────────────────────────────────────────────────
m = folium.Map(
    location=[42.31, -71.08],
    zoom_start=12,
    tiles="CartoDB positron",
)

# ── Census tract choropleth (green → red, matching StoryMap) ──
tracts_geojson = json.loads(gdf_tracts.to_json())
for i, feat in enumerate(tracts_geojson["features"]):
    feat["id"] = gdf_tracts.iloc[i]["tract_id"]
    feat["properties"]["shoot_count"] = int(gdf_tracts.iloc[i]["shoot_count"])
    feat["properties"]["tract_id"] = gdf_tracts.iloc[i]["tract_id"]

# StoryMap-style bins: 0, 1, 2-3, 4-5, 6+
# Custom style function to replicate the green-yellow-orange-red scale
def style_function(feature):
    count = feature["properties"].get("shoot_count", 0)
    if count == 0:
        fill = "#2d8c4e"    # dark green
    elif count == 1:
        fill = "#78b85e"    # green
    elif count <= 3:
        fill = "#c8d96f"    # yellow-green
    elif count <= 10:
        fill = "#ffe08b"    # yellow
    elif count <= 25:
        fill = "#fdbe6f"    # light orange
    elif count <= 50:
        fill = "#f4845f"    # orange-red
    elif count <= 100:
        fill = "#e8533e"    # red
    else:
        fill = "#c0233c"    # dark red
    return {
        "fillColor": fill,
        "fillOpacity": 0.70,
        "color": "#bbb",
        "weight": 0.8,
    }

folium.GeoJson(
    tracts_geojson,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["tract_id", "shoot_count"],
        aliases=["Census Tract:", "Total Shootings:"],
        style="font-family:system-ui;font-size:12px;",
    ),
    name="Shootings by census tract",
).add_to(m)

print(f"✓ Choropleth: {len(tracts_geojson['features'])} census tracts")

In [ ]:
# ── Crew-to-crew affiliations (solid, muted blue) ─────────────
affiliations_fg = folium.FeatureGroup(name="Affiliations", show=True)
crew_loc = {
    row["Crew"]: (row["Lat"], row["Long"])
    for _, row in df_crews.iterrows()
}

for a, b in crew_affiliations:
    if a in crew_loc and b in crew_loc:
        folium.PolyLine(
            locations=[crew_loc[a], crew_loc[b]],
            color="#56B4E9",
            weight=2.5,
            opacity=0.7,
            tooltip=f"Affiliation: {a} ↔ {b}",
        ).add_to(affiliations_fg)
affiliations_fg.add_to(m)

# ── Crew-to-crew conflicts (dashed vermilion) ────────────────
conflicts_fg = folium.FeatureGroup(name="Conflicts", show=True)

for a, b in crew_conflicts:
    if a in crew_loc and b in crew_loc:
        folium.PolyLine(
            locations=[crew_loc[a], crew_loc[b]],
            color="#D55E00",
            weight=2.5,
            opacity=0.7,
            dash_array="10,6",
            tooltip=f"Conflict: {a} ↔ {b}",
        ).add_to(conflicts_fg)
conflicts_fg.add_to(m)

# ── Catalyst-to-catalyst links (dotted grey) ─────────────────
catalyst_net_fg = folium.FeatureGroup(name="Catalyst links", show=True)

def jitter(lat, lon, seed_key, radius=0.003):
    rnd = random.Random(seed_key)
    angle = rnd.uniform(0, 2 * math.pi)
    r = rnd.uniform(0.3, 1.0) * radius
    return (lat + r * math.cos(angle), lon + r * math.sin(angle))

catalyst_loc = {}
for cat, (crew, lat, lon) in catalyst_to_crew.items():
    catalyst_loc[cat] = jitter(lat, lon, cat)

for a, b in catalyst_edges:
    if a in catalyst_loc and b in catalyst_loc:
        folium.PolyLine(
            locations=[catalyst_loc[a], catalyst_loc[b]],
            color="#888",
            weight=1.2,
            opacity=0.4,
            dash_array="2,5",
            tooltip=f"Network: {a} ↔ {b}",
        ).add_to(catalyst_net_fg)
catalyst_net_fg.add_to(m)

print(f"✓ Arcs: {len(crew_affiliations)} affiliations, "
      f"{len(crew_conflicts)} conflicts, {len(catalyst_edges)} catalyst links")

In [ ]:
# ── Crew markers (orange graduated circles — StoryMap style) ──
crews_fg = folium.FeatureGroup(name="Crews & catalysts", show=True)

CREW_ORANGE = "#E8782A"

for _, row in df_crews.iterrows():
    crew = row["Crew"]
    n_cats = len(crew_catalysts.get(crew, []))

    # Graduated radius matching StoryMap legend
    if n_cats > 12:
        radius = 22
    elif n_cats >= 9:
        radius = 18
    elif n_cats >= 6:
        radius = 14
    elif n_cats >= 4:
        radius = 10
    else:
        radius = 6

    popup_html = (
        f'<div style="font-family:system-ui;min-width:180px;">'
        f'<div style="font-size:14px;font-weight:bold;color:{CREW_ORANGE};">'
        f'{crew}</div>'
        f'<div style="font-size:11px;color:#555;margin-top:4px;">'
        f'{n_cats} catalyst(s)</div>'
        f'<div style="font-size:11px;margin-top:6px;"><b>Members:</b><br>'
        + "<br>".join(crew_catalysts.get(crew, []))
        + '</div></div>'
    )

    folium.CircleMarker(
        location=[row["Lat"], row["Long"]],
        radius=radius,
        color="#c45a1a",
        weight=2,
        fill=True,
        fill_color=CREW_ORANGE,
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=f"{crew} ({n_cats} catalysts)",
    ).add_to(crews_fg)

    folium.Marker(
        location=[row["Lat"], row["Long"]],
        icon=folium.DivIcon(
            html=(
                f'<div style="font-size:11px;color:#333;font-weight:600;'
                f'text-shadow:0 0 3px #fff, 0 0 6px #fff;'
                f'white-space:nowrap;transform:translate(16px,-6px);">'
                f'{crew}</div>'
            ),
            icon_size=(120, 18),
            icon_anchor=(0, 0),
        ),
    ).add_to(crews_fg)

# ── Individual catalyst dots ─────────────────────────────────
for cat, (lat, lon) in catalyst_loc.items():
    folium.CircleMarker(
        location=[lat, lon],
        radius=3,
        color="#c45a1a",
        weight=0.6,
        fill=True,
        fill_color=CREW_ORANGE,
        fill_opacity=0.8,
        tooltip=f"{cat} ({catalyst_to_crew[cat][0]})",
    ).add_to(crews_fg)

crews_fg.add_to(m)
print(f"✓ Markers: {len(df_crews)} crews, {len(catalyst_loc)} catalysts")

In [ ]:
# ── Title bar ─────────────────────────────────────────────────
date_min = df_shoot["shooting_date"].min().strftime("%b %Y")
date_max = df_shoot["shooting_date"].max().strftime("%b %Y")

title_html = f"""
<div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);z-index:9999;
     background:rgba(255,255,255,0.94);padding:8px 20px;border-radius:4px;
     font-family:'Helvetica Neue',system-ui,sans-serif;color:#222;
     font-size:14px;font-weight:600;
     box-shadow:0 1px 4px rgba(0,0,0,0.12);border:1px solid #ddd;">
    Boston Total Shootings {date_min} to {date_max}
    <span style="color:#888;font-weight:400;">&nbsp;·&nbsp;N = {len(df_shoot):,}</span>
    &nbsp;&amp;&nbsp;
    <span style="color:{CREW_ORANGE};font-weight:600;">Crews</span>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

# ── Legend (left side, matching StoryMap layout) ──────────────
legend_html = f"""
<div style="position:fixed;bottom:40px;left:20px;z-index:9999;
     background:rgba(255,255,255,0.94);padding:14px 18px;border-radius:4px;
     font-family:'Helvetica Neue',system-ui,sans-serif;color:#333;
     font-size:11px;line-height:1.6;max-width:240px;
     box-shadow:0 1px 4px rgba(0,0,0,0.12);border:1px solid #ddd;">

    <div style="font-size:13px;font-weight:700;margin-bottom:6px;">
      Boston Crews</div>
    <div style="font-size:10px;color:#666;margin-bottom:4px;">Number of Catalysts</div>
    <div style="display:flex;align-items:center;gap:8px;margin-top:3px;">
      <div style="width:28px;height:28px;border-radius:50%;background:{CREW_ORANGE};
           border:2px solid #c45a1a;opacity:0.85;"></div>
      <span>&gt; 12</span></div>
    <div style="display:flex;align-items:center;gap:8px;margin-top:3px;">
      <div style="width:22px;height:22px;border-radius:50%;background:{CREW_ORANGE};
           border:2px solid #c45a1a;opacity:0.85;"></div>
      <span>9</span></div>
    <div style="display:flex;align-items:center;gap:8px;margin-top:3px;">
      <div style="width:16px;height:16px;border-radius:50%;background:{CREW_ORANGE};
           border:2px solid #c45a1a;opacity:0.85;"></div>
      <span>6</span></div>
    <div style="display:flex;align-items:center;gap:8px;margin-top:3px;">
      <div style="width:12px;height:12px;border-radius:50%;background:{CREW_ORANGE};
           border:2px solid #c45a1a;opacity:0.85;"></div>
      <span>4</span></div>
    <div style="display:flex;align-items:center;gap:8px;margin-top:3px;">
      <div style="width:8px;height:8px;border-radius:50%;background:{CREW_ORANGE};
           border:2px solid #c45a1a;opacity:0.85;"></div>
      <span>&lt; 1</span></div>

    <div style="margin-top:10px;padding-top:8px;border-top:1px solid #ddd;
         font-size:13px;font-weight:700;margin-bottom:4px;">
      Shootings by Census Tract</div>
    <div style="font-size:10px;color:#666;margin-bottom:4px;">
      Number of Total Shootings (Fatal &amp; Nonfatal)</div>

    <div style="display:flex;align-items:center;gap:6px;margin-top:3px;">
      <div style="width:18px;height:14px;background:#2d8c4e;border:1px solid #bbb;"></div>
      <span>0</span></div>
    <div style="display:flex;align-items:center;gap:6px;margin-top:2px;">
      <div style="width:18px;height:14px;background:#78b85e;border:1px solid #bbb;"></div>
      <span>1</span></div>
    <div style="display:flex;align-items:center;gap:6px;margin-top:2px;">
      <div style="width:18px;height:14px;background:#c8d96f;border:1px solid #bbb;"></div>
      <span>2 – 3</span></div>
    <div style="display:flex;align-items:center;gap:6px;margin-top:2px;">
      <div style="width:18px;height:14px;background:#ffe08b;border:1px solid #bbb;"></div>
      <span>4 – 10</span></div>
    <div style="display:flex;align-items:center;gap:6px;margin-top:2px;">
      <div style="width:18px;height:14px;background:#fdbe6f;border:1px solid #bbb;"></div>
      <span>11 – 25</span></div>
    <div style="display:flex;align-items:center;gap:6px;margin-top:2px;">
      <div style="width:18px;height:14px;background:#f4845f;border:1px solid #bbb;"></div>
      <span>26 – 50</span></div>
    <div style="display:flex;align-items:center;gap:6px;margin-top:2px;">
      <div style="width:18px;height:14px;background:#e8533e;border:1px solid #bbb;"></div>
      <span>51 – 100</span></div>
    <div style="display:flex;align-items:center;gap:6px;margin-top:2px;">
      <div style="width:18px;height:14px;background:#c0233c;border:1px solid #bbb;"></div>
      <span>100+</span></div>

    <div style="margin-top:10px;padding-top:8px;border-top:1px solid #ddd;">
      <div style="display:flex;align-items:center;gap:6px;margin-top:2px;">
        <svg width="20" height="4"><line x1="0" y1="2" x2="20" y2="2"
          stroke="#56B4E9" stroke-width="2.5"/></svg>
        <span>Affiliation</span></div>
      <div style="display:flex;align-items:center;gap:6px;margin-top:2px;">
        <svg width="20" height="4"><line x1="0" y1="2" x2="20" y2="2"
          stroke="#D55E00" stroke-width="2.5" stroke-dasharray="4,3"/></svg>
        <span>Conflict</span></div>
      <div style="display:flex;align-items:center;gap:6px;margin-top:2px;">
        <svg width="20" height="4"><line x1="0" y1="2" x2="20" y2="2"
          stroke="#888" stroke-width="1.5" stroke-dasharray="2,4"/></svg>
        <span>Catalyst link</span></div>
    </div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=True).add_to(m)

m

## 6. Export

In [ ]:
m.save("boston_crew_network_dashboard.html")
print("✓ Saved → boston_crew_network_dashboard.html")
print()
print("To refresh data: set USE_LOCAL = False and re-run all cells.")
print("Boston PD updates weekly with a 7-day delay.")
print()
print("To automate weekly refresh with cron + papermill:")
print("  0 6 * * MON papermill boston_crew_dashboard.ipynb out.ipynb")